In [1]:
import numpy as np
import pandas as pd
import os
import shutil
import matplotlib.pyplot as plt
import datetime
import cv2 as cv
from sklearn.metrics import r2_score
from scipy.stats import pearsonr,spearmanr
import matplotlib.dates as mdates
import datetime
from sklearn.decomposition import PCA, KernelPCA
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split, GridSearchCV
import joblib as jb
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, AdaBoostRegressor, BaggingRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.cluster import SpectralClustering, MiniBatchKMeans, SpectralBiclustering, SpectralCoclustering
from sklearn.linear_model import SGDRegressor
from sklearn.linear_model import Ridge, Lasso
import cv2 as cv
from skimage.metrics import structural_similarity
import matplotlib.dates as mdates


In [2]:
#INPUT CSV DATA
belur=pd.read_csv("combined_dataset.csv")

In [3]:
#CONVERTING BELUR DAY,MONTH,HOUR,MIN INTO CYCLIC DATA
belur['Day_sin'] = np.sin(2 * np.pi * belur['Day']/31)
belur['Day_cos'] = np.cos(2 * np.pi * belur['Day']/31)
belur['Mon_sin'] = np.sin(2 * np.pi * belur['Month']/12)
belur['Mon_cos'] = np.cos(2 * np.pi * belur['Month']/12)
belur['Hour_sin'] = np.sin(2 * np.pi * belur['Hour']/24)
belur['Hour_cos'] = np.cos(2 * np.pi * belur['Hour']/24)
belur['Min_sin'] = np.sin(2 * np.pi * belur['Min']/60)
belur['Min_cos'] = np.cos(2 * np.pi * belur['Min']/60)

In [4]:
#SPLIT BELUR DATA
train_b,test_b = train_test_split(belur,test_size=0.2,random_state=42)

In [5]:
#NORMALIZING BELUR PM2.5 VALUES
train_b['PM2.5_Norm'] = (train_b['PM2.5'] - train_b['PM2.5'].min())/(train_b['PM2.5'].max() - train_b['PM2.5'].min())
test_b['PM2.5_Norm'] = (test_b['PM2.5'] - train_b['PM2.5'].min())/(train_b['PM2.5'].max() - train_b['PM2.5'].min())

In [6]:
#CREATING X1 LIST FOR BELUR DATA
X1_train = train_b[['Belur_SSIM','Belur_ERODE','FRB_SSIM','FRB_ERODE','Day_sin','Day_cos','Mon_sin','Mon_cos','Hour_sin','Hour_cos']]
X1_test = test_b[['Belur_SSIM','Belur_ERODE','FRB_SSIM','FRB_ERODE','Day_sin','Day_cos','Mon_sin','Mon_cos','Hour_sin','Hour_cos']] 


In [7]:
#CREATING Y1 LIST FOR BELUR DATA
Y1_train= train_b['PM2.5_Norm']
Y1_test= test_b['PM2.5_Norm']

In [8]:
#CREATING HYPER_PARAMETER GRID
param_grid = {
    'n_estimators': np.arange(10,1020,10),
    'max_features': ['sqrt','log2','sqrt','log2',0.1,0.2,0.3,0.5,0.7,1.0],
    'bootstrap': [True,False]
}

In [9]:
#GRID SEARCH FOR BELUR DATA
#CREATE GRID SEARCH
grid = GridSearchCV(ExtraTreesRegressor(),param_grid=param_grid,refit=True,verbose=3,n_jobs=-1)

In [10]:
#TRAIN GRID SEARCH
grid.fit(X1_train,Y1_train)

Fitting 5 folds for each of 2020 candidates, totalling 10100 fits
[CV 5/5] END bootstrap=True, max_features=sqrt, n_estimators=10;, score=0.641 total time=   0.0s
[CV 2/5] END bootstrap=True, max_features=sqrt, n_estimators=40;, score=0.775 total time=   0.1s
[CV 4/5] END bootstrap=True, max_features=sqrt, n_estimators=50;, score=0.791 total time=   0.1s
[CV 5/5] END bootstrap=True, max_features=sqrt, n_estimators=60;, score=0.782 total time=   0.1s
[CV 3/5] END bootstrap=True, max_features=sqrt, n_estimators=80;, score=0.793 total time=   0.1s
[CV 3/5] END bootstrap=True, max_features=sqrt, n_estimators=100;, score=0.796 total time=   0.2s
[CV 2/5] END bootstrap=True, max_features=sqrt, n_estimators=120;, score=0.822 total time=   0.2s
[CV 3/5] END bootstrap=True, max_features=sqrt, n_estimators=120;, score=0.801 total time=   0.2s
[CV 3/5] END bootstrap=True, max_features=sqrt, n_estimators=150;, score=0.800 total time=   0.2s
[CV 4/5] END bootstrap=True, max_features=sqrt, n_estimat

,estimator,ExtraTreesRegressor()
,param_grid,"{'bootstrap': [True, False], 'max_features': ['sqrt', 'log2', ...], 'n_estimators': array([ 10, ... 1000, 1010])}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,np.int64(160)


In [11]:
grid.best_params_

{'bootstrap': False, 'max_features': 1.0, 'n_estimators': np.int64(160)}

In [12]:
grid.score(X1_train,Y1_train)

1.0

In [13]:
grid.score(X1_test,Y1_test)

0.929847079694157

In [14]:
#BUILD EXTRA TREE REGRESSOR MODELS FOR BELUR DATA
etr_belur = ExtraTreesRegressor(bootstrap = True, max_features = 1.0, n_estimators = 410, random_state=42)
etr_belur.fit(X1_train,Y1_train)

,n_estimators,410
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [15]:
etr_belur.score(X1_train,Y1_train)

0.9862169671327661

In [16]:
etr_belur.score(X1_test,Y1_test)

0.904476377974552

In [17]:
#SAVE THE MODELS AS PICKLE FILES
jb.dump(etr_belur, 'etr_frb_belur.pkl')

['etr_frb_belur.pkl']

In [18]:
#PREDICT PM2.5 FOR BELUR DATA
predtrain_b = etr_belur.predict(X1_train)
predtest_b = etr_belur.predict(X1_test)

In [19]:
train_b['ETR_ROI'] = predtrain_b * (train_b['PM2.5'].max() - train_b['PM2.5'].min()) + train_b['PM2.5'].min()
test_b['ETR_ROI'] = predtest_b * (train_b['PM2.5'].max() - train_b['PM2.5'].min()) + train_b['PM2.5'].min()

In [20]:
#SAVE AS CSV
#BELUR
train_b.to_csv('frb_belur_train.csv',index=False)
test_b.to_csv('frb_belur_test.csv',index=False)

In [22]:
r2_score(test_b['PM2.5'],test_b['ETR_ROI'])

0.904476377974552